# VLA-0-Smol LIBERO-Goal baseline

10 episodes each of plate and stove, float32: first with no ensembling, then with the model card's ensembling.

Settings: **Accelerator = GPU T4 x2** and **Internet = on**.


In [ ]:
!nvidia-smi -L


In [ ]:
!pip install -q "cmake<4"
!pip install -q "lerobot[vla0_smol,libero] @ git+https://github.com/Robot-Learning-Collective/lerobot-experiments.git@8e71efbebbb7ea2942b579b1d21e3ec11ac2d271" pyyaml


In [ ]:
import json, pathlib
files = json.loads("{\"scripts/eval_libero.py\": \"\\\"\\\"\\\"Start lerobot-eval with env.task_ids working, without changing the installed package.\\n\\nTwo problems at the pinned commit:\\n1. The field is typed Iterable[int], which the command-line parser can't read.\\n2. LiberoEnv.gym_kwargs hard-codes \\\"task_ids\\\": [1], so the option would be ignored and every\\n   eval would silently run task 1 of the suite.\\n\\\"\\\"\\\"\\nfrom collections.abc import Iterable\\n\\nimport draccus\\nimport yaml\\n\\nfrom lerobot.envs.configs import LiberoEnv\\n\\n\\ndef decode_int_list(raw_value, path):\\n    values = yaml.safe_load(raw_value) if isinstance(raw_value, str) else raw_value\\n    return [int(value) for value in values]\\n\\n\\ndef gym_kwargs(self):\\n    if not self.task_ids:\\n        raise ValueError(\\\"Pass --env.task_ids explicitly, for example --env.task_ids=[8]\\\")\\n    return {\\\"obs_type\\\": self.obs_type, \\\"render_mode\\\": self.render_mode, \\\"task_ids\\\": list(self.task_ids)}\\n\\n\\ndraccus.decode.register(Iterable[int], decode_int_list)\\nLiberoEnv.gym_kwargs = property(gym_kwargs)\\n\\nif __name__ == \\\"__main__\\\":\\n    from lerobot.scripts.lerobot_eval import main\\n\\n    main()\\n\", \"config/smoke.yaml\": \"# Stage 0 smoke run: prompt-only forward pass, no generation, no simulator.\\nname: smoke\\npins: &pins\\n  lerobot_commit: 8e71efbebbb7ea2942b579b1d21e3ec11ac2d271   # lerobot-experiments, dev\\n  model: Robot-Learning-Collective/VLA-0-Smol\\n  model_revision: 551fe3d5d3debb6532f0675a7a68b3b245ead4d8\\n  base_vlm: HuggingFaceTB/SmolVLM2-500M-Video-Instruct       # processor and tokenizer come from here\\n  base_vlm_revision: 7b375e1b73b11138ff12fe22c8f2822d8fe03467\\n  dataset: HuggingFaceVLA/libero\\n  dataset_revision: 86958911c0f959db2bbbdb107eb3e17c5f9c798e\\n\\n# Recorded demo frames, not saved simulator states. Frame position is a placeholder\\n# and is NOT checked to be after the bowl is lifted. Stage 2 replaces this.\\nsnapshots:\\n  source: dataset_frame\\n  task: put the bowl on the plate\\n  n_episodes: 1\\n  frame_fraction: 0.6\\n\\ninstructions:\\n  plate: put the bowl on the plate\\n  stove: put the bowl on the stove\\n\\nreadout:\\n  words: [plate, stove]   # exact tokens get checked and logged by the run\\n  layers: all\\n  positions: [prompt_end]\\n\\nchecks:\\n  self_patch_identity: true\\n  self_patch_layers: all\\n\\ndevice: cpu\\nseed: 0\\n\"}")
for name, text in files.items():
    path = pathlib.Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
print(sorted(files))


In [ ]:
import os, subprocess, sys
test = """
import sys
sys.path.insert(0, "scripts")
import eval_libero
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env
for task_id in (8, 1):
    env = make_env(LiberoEnv(task="libero_goal", task_ids=[task_id]), n_envs=1)["libero_goal"][task_id]
    observation, _ = env.reset(seed=1000)
    print("TEST task", task_id, env.call("task_description"), observation["pixels"]["image"].shape)
    env.close()
print("TEST PASSED")
"""
subprocess.run("echo N | python -c 'import libero.libero'", shell=True)
result = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True)
print(result.stdout[-1500:])
print(result.stderr[-3000:])
assert "TEST PASSED" in result.stdout, "LIBERO did not start on this machine"


In [ ]:
import os, shutil, subprocess, sys, time
import torch
import yaml
from huggingface_hub import snapshot_download

pins = yaml.safe_load(open("config/smoke.yaml"))["pins"]
model_dir = snapshot_download(pins["model"], revision=pins["model_revision"])
base_dir = snapshot_download(
    pins["base_vlm"],
    revision=pins["base_vlm_revision"],
    allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja"],
)
n_gpus = torch.cuda.device_count()
print("GPUs:", n_gpus, [torch.cuda.get_device_name(i) for i in range(n_gpus)])
TASKS = {"plate": 8, "stove": 1}
EPISODES = 10


def run_phase(phase, n_action_steps, ensemble_size):
    started = time.time()
    jobs = []
    for index, (name, task_id) in enumerate(TASKS.items()):
        output = f"stage1/{phase}/{name}_{EPISODES}ep"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        command = [
            sys.executable, "scripts/eval_libero.py",
            f"--policy.path={model_dir}",
            f"--policy.vlm_checkpoint={base_dir}",
            "--policy.device=cuda",
            "--policy.use_amp=false",
            f"--policy.n_action_steps={n_action_steps}",
            f"--policy.ensemble_size={ensemble_size}",
            "--env.type=libero",
            "--env.task=libero_goal",
            f"--env.task_ids=[{task_id}]",
            "--eval.batch_size=1",
            f"--eval.n_episodes={EPISODES}",
            f"--output_dir={output}",
        ]
        environment = dict(os.environ, CUDA_VISIBLE_DEVICES=str(index % max(n_gpus, 1)))
        log = open(f"{output}.log", "w")
        jobs.append((name, output, subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, env=environment)))
    for name, output, job in jobs:
        code = job.wait()
        print(phase, name, "exit", code, "elapsed so far", round((time.time() - started) / 60, 1), "min")
        if code != 0:
            print(open(f"{output}.log", errors="ignore").read()[-3000:])
        else:
            import json
            overall = json.load(open(f"{output}/eval_info.json"))["overall"]
            print(phase, name, {k: overall[k] for k in ("pc_success", "n_episodes", "eval_s", "eval_ep_s")})
    shutil.make_archive("/kaggle/working/stage1_results", "zip", "stage1")
    print("saved stage1_results.zip after", phase)


# Cheaper executor first, so a usable result exists early; then the model card's ensembling.
run_phase("no_ensembling", n_action_steps=8, ensemble_size=0)
run_phase("ensembling", n_action_steps=0, ensemble_size=8)


In [ ]:
!pip freeze > /kaggle/working/stage1_pip_freeze.txt
!ls -la /kaggle/working
